In [ ]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import r2_score
import torch.nn.functional as F
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter

In [ ]:
# Functions

def estimate_lognormal_params(mean, median):
    sigma = np.sqrt(2 * np.log(mean / median))
    mu = np.log(median)
    return mu, sigma

# Function to generate synthetic data using log-normal distribution and clip to observed range
def generate_lognormal_samples(mean, median, min_val, max_val, size=10000):
    mu, sigma = estimate_lognormal_params(mean, median)
    samples = np.random.lognormal(mean=mu, sigma=sigma, size=size)
    return np.clip(samples, min_val, max_val)

def generate_time (t_initial, t_final, n_timesteps):
    t_numpy = np.linspace(t_initial, t_final, n_timesteps)
    t_reshape = t_numpy.reshape(1, -1)
    t_torch = torch.tensor(t_reshape, dtype=torch.float32).to(device)
    return t_numpy, t_reshape, t_torch

def smooth_peak(curve, window_length=11, polyorder=3, baseline_ratio=0.01):
    curve = np.asarray(curve)
    smoothed = savgol_filter(curve, window_length=window_length, polyorder=polyorder)

    peak_idx = np.argmax(smoothed)
    peak_val = smoothed[peak_idx]
    threshold = peak_val * baseline_ratio

    # Find start (left of peak)
    start_idx = 0
    for i in range(peak_idx, 0, -1):
        if smoothed[i] < threshold:
            start_idx = i
            break

    # Find end (right of peak)
    end_idx = len(curve) - 1
    for i in range(peak_idx, len(curve)):
        if smoothed[i] < threshold:
            end_idx = i
            break

    # Keep only the main peak region
    cleaned = np.zeros_like(curve)
    cleaned[start_idx:end_idx + 1] = curve[start_idx:end_idx + 1]

    return cleaned


def zscore(x, eps=1e-12):
    x = np.asarray(x, dtype=float)
    mu = x.mean()
    sd = x.std()
    if sd < eps:
        return x * 0.0
    return (x - mu) / sd

def best_lag_correlation(a, b, max_lag=None, normalize='zscore'):

    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    T = min(len(a), len(b))
    a = a[:T]
    b = b[:T]

    if normalize == 'zscore':
        a_n = zscore(a)
        b_n = zscore(b)
    else:
        a_n = a
        b_n = b

    if max_lag is None:
        max_lag = T - 1

    best_corr = -np.inf
    best_lag = 0
    best_pair = (None, None)

    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:

            a_slice = a_n[-lag:]
            b_slice = b_n[:T+lag]
        elif lag > 0:

            a_slice = a_n[:T-lag]
            b_slice = b_n[lag:]
        else:
            a_slice = a_n
            b_slice = b_n

        if len(a_slice) < 2:
            continue

        corr = np.dot(a_slice, b_slice) / (len(a_slice) - 1)

        if corr > best_corr:
            best_corr = corr
            best_lag = lag
            if lag < 0:
                a_raw = a[-lag:]
                b_raw = b[:T+lag]
            elif lag > 0:
                a_raw = a[:T-lag]
                b_raw = b[lag:]
            else:
                a_raw = a
                b_raw = b
            best_pair = (a_raw, b_raw)

    return {'lag': best_lag, 'corr': best_corr, 'a_seg': best_pair[0], 'b_seg': best_pair[1]}

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Uniform channel test
# Generate synthetic values for each variable
mass = np.array([9.62])
area = np.array([224])
velocity = np.array([0.51])
distance = np.array([10000, 20000, 50000, 100000])
dispersion = np.array([130])

# Linear scale and reshape
M = np.repeat(mass, 4).reshape(-1,1)
A = np.repeat(area, 4).reshape(-1,1)
x = distance.reshape(-1,1)
u =np.repeat(velocity, 4).reshape(-1,1)
Q = np.array(A*u)
Q = Q.reshape(-1,1)
R= 1
theta = 1
D = np.repeat(dispersion, 4).reshape(-1,1)

n_timesteps = 100000
t_numpy, t_reshape, t_torch = generate_time(1, 3e6, n_timesteps)
c_downstream_filter = (1e6 * M) / (2 * theta * A * R * np.sqrt(np.pi * D * t_reshape/ R)) * np.exp(-((x - u * t_reshape / R) ** 2) / (4 * D * t_reshape / R))

t_peak = []
max_concentrion_array = []
for i in range (c_downstream_filter.shape[0]):
  max_concentrion_idx = np.argmax(c_downstream_filter[i,:])
  max_concentrion = np.max(c_downstream_filter[i,:])
  time_peak = t_reshape[:,max_concentrion_idx]
  t_peak = np.append(t_peak,time_peak)
  max_concentrion_array = np.append(max_concentrion_array,max_concentrion)

v = distance/t_peak
pe = (distance*velocity)/dispersion
valid_v = (v >= 0.1) & (v <= 6)
valid_pe = (pe >= 10) & (pe <= 1500)
valid_max_c = (max_concentrion_array >= 0.3) & (max_concentrion_array <= 3000)
valid_ratio =  valid_v & valid_pe & valid_max_c

In [ ]:
# Linear scale and reshape
valid_M = M[valid_ratio]
valid_A = A[valid_ratio]
valid_x = x
x_upstream = np.array(valid_x/ 10)
x_upstream = x_upstream.reshape(-1,1)
valid_u = u[valid_ratio]
valid_D = D[valid_ratio]
valid_Q = np.array(valid_A*valid_u)

In [ ]:
# Calculate upstream and downstream BTCs using ADE analytical solution
c_upstream = (1e6 * valid_M) / (2 * theta * valid_A * R * np.sqrt(np.pi * valid_D * t_reshape / R)) * np.exp(-((x_upstream - valid_u * t_reshape / R) ** 2) / (4 * valid_D * t_reshape / R))

c_downstream = (1e6 * valid_M) / (2 * theta * valid_A * R * np.sqrt(np.pi * valid_D * t_reshape/ R)) * np.exp(-((valid_x - valid_u * t_reshape / R) ** 2) / (4 * valid_D * t_reshape / R))

In [ ]:
dtype = torch.float32
input = torch.tensor(c_upstream).to(device,dtype)
target = torch.tensor(c_downstream).to(device,dtype)
input_test = input
target_test = target

In [ ]:
input_size = input.shape[1]
output_size = target.shape[1]

# Define the MLP model
class MLP(nn.Module):
    def __init__(self, input_size, output_size):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 2048),
            nn.ReLU(),
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Linear(1024, output_size)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
# Import model
checkpoint = torch.load("/content/drive/MyDrive/model_checkpoint_pretrained_test.pth", map_location='cpu')
model = MLP(input_size, output_size).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)

In [ ]:
model.eval()
# Lists to store actual and predicted values
actual_curves = []
predicted_curves = []
with torch.no_grad():  # Disable gradient calculations
    outputs_testing = model(input_test)  # Get predictions

    actual_curves.extend(target_test.cpu().numpy())
    predicted_curves.extend(outputs_testing.cpu().numpy())

actual_curves = np.array(actual_curves)
predicted_curves = np.array(predicted_curves)
predicted_curves[predicted_curves < 0] = 0

In [ ]:
max_lag = 100
use_normalize = 'zscore'

cc_results = []
for i in range(actual_curves.shape[0]):
    a = actual_curves[i]
    b = smooth_peak(predicted_curves[i])
    res = best_lag_correlation(a, b, max_lag=max_lag, normalize=use_normalize)
    cc_results.append(res)

cc_mean_corr = np.mean([r['corr'] for r in cc_results])
print(f"[Cross-Correlation] Mean correlation: {cc_mean_corr:.4f}")

In [ ]:
num_curves_to_plot = len(actual_curves)

for i in range(num_curves_to_plot):
    plt.figure(figsize=(8, 6))

    plt.semilogx(t_numpy, actual_curves[i], label=f'True {i}', linestyle='dashed')
    plt.semilogx(t_numpy, smooth_peak(predicted_curves[i]), label=f'Predicted {i}', linestyle='solid')

    plt.grid(True)
    plt.xlabel('Time Index')
    plt.ylabel('Concentration (ppb)')
    plt.title('Tested Breakthrough Curves: True vs Predicted')

    textstr = '\n'.join([
        f'Mass = {mass[0]} kg',
        f'Area = {area[0]} m²',
        f'Velocity = {velocity[0]} m/s',
        f'Dispersion = {dispersion[0]} m²/s',
        f'Distance = {distance[i]} m'
    ])

    plt.gca().text(
        0.02, 0.98, textstr,
        transform=plt.gca().transAxes,
        fontsize=10,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
    )

    plt.legend()
    plt.tight_layout()
    plt.show()